<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 1 — Contexte métier & Exploration des données
### 📝 VERSION APPRENANT

> **Comment utiliser ce notebook :**  
> Les blocs `MÉTHODE` t'expliquent quoi faire et pourquoi.  
> Les cellules `# 📝 TODO` sont à compléter.  
> Les sections `### 🧠 Tes observations` t'invitent à interpréter tes résultats.

| | |
|---|---|
| **Entreprise** | MarketPulse — agence marketing digital, Afrique de l'Ouest |
| **Outils** | Python, DuckDB (JupySQL), matplotlib |
| **Durée estimée** | 3h à 4h |

> **Contexte :** MarketPulse gère les campagnes publicitaires de 25 clients sur 4 canaux (Facebook, Google, Email, SMS) dans 6 pays. Ton objectif est de comprendre la structure des données, détecter les anomalies et calculer les premiers KPIs marketing.

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8',
                      'axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 {"Colab" if IN_COLAB else "Local"} — {SAVE_PATH}')

---
## 1. Connexion DuckDB et chargement des 7 tables

### MÉTHODE
`read_csv_auto()` charge les CSV directement depuis GitHub, infère les types et crée les tables en mémoire. Avantage : tu peux écrire du SQL analytique (window functions, CTEs) directement sur ces tables.

> **Ordre de chargement :** tables de référence d'abord (`clients_agence`, `audiences`, `creatifs`), puis tables de faits (`campagnes`, `performances_daily`, `conversions`, `contacts_crm`).

In [ ]:
# 📝 TODO : Créer une connexion DuckDB et charger les 7 tables CSV depuis GitHub avec read_csv_auto()
# 💡 Indice 1 : conn = duckdb.connect()
# 💡 Indice 2 : BASE_URL = 'https://raw.githubusercontent.com/.../marketpulse/data/'
# 💡 Indice 3 : conn.execute("CREATE TABLE nom AS SELECT * FROM read_csv_auto('URL')")
# 💡 Indice 4 : Tables à charger : clients_agence, audiences, creatifs, campagnes, performances_daily, conversions, contacts_crm

BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

# TODO : créer la connexion et charger les 7 tables
conn = ...


### MÉTHODE
Active JupySQL pour écrire des requêtes avec `%%sql` directement dans les cellules.

In [ ]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## 2. Exploration des tables et du modèle de données

### MÉTHODE
Le modèle MarketPulse est un schéma en étoile autour de `campagnes` :

```
clients_agence ──────┐
audiences ───────────┤──► campagnes ──► performances_daily
creatifs ────────────┘         │
                               └──► conversions → contacts_crm
```

> **Point d'attention :** `campagnes.client_id` = le client de l'agence (Orange CI). `contacts_crm.client_id` pointe aussi vers `clients_agence` mais représente les clients *finaux* d'Orange CI, pas Orange CI elle-même.

In [ ]:
%%sql df_schema <<
-- 📝 TODO : Afficher le schéma de toutes les tables : table_name, column_name, data_type
-- 💡 Indice 1 : Table à interroger : information_schema.columns
-- 💡 Indice 2 : Filtrer sur table_schema = 'main'
-- 💡 Indice 3 : Trier par table_name et ordinal_position


In [ ]:
%%sql  <<
-- 📝 TODO : Afficher les 5 premières lignes de la table campagnes
-- 💡 Indice 1 : SELECT * FROM campagnes LIMIT ...


In [ ]:
%%sql  <<
-- 📝 TODO : Afficher les clients triés par budget_mensuel_fcfa décroissant
-- 💡 Indice 1 : SELECT les colonnes utiles : client_id, nom, secteur, budget_mensuel_fcfa, objectif_principal


### 🧠 Tes observations

> - Combien de colonnes contient `performances_daily` ? Lesquelles sont des métriques calculées ?
> - Quelle est la plage de budgets mensuels des clients ? Que cela implique-t-il pour les analyses en valeur absolue ?
> - Pourquoi ne faut-il pas utiliser `client_id` de `contacts_crm` pour identifier les clients de l'agence ?


---
## 3. Diagnostic qualité — Les 6 anomalies

### MÉTHODE
Les données publicitaires multi-plateformes sont sujettes aux anomalies car elles transitent par des APIs tierces. Trois types courants :
1. **Incohérences calculées** — CTR ≠ clics/impressions
2. **Valeurs physiquement impossibles** — clics > impressions
3. **Erreurs de signe** — revenus négatifs

> **Règle fondamentale :** ne jamais corriger sans comprendre l'origine.

In [ ]:
%%sql df_diag_perfs <<
-- 📝 TODO : Compter les 5 types d'anomalies dans performances_daily en une seule requête
-- 💡 Indice 1 : COUNT(*) AS total_lignes
-- 💡 Indice 2 : SUM(CASE WHEN clics > impressions THEN 1 ELSE 0 END) AS clics_sup_impressions
-- 💡 Indice 3 : SUM(CASE WHEN revenu_genere_fcfa < 0 THEN 1 ELSE 0 END) AS revenu_negatif
-- 💡 Indice 4 : SUM(CASE WHEN roas > 50 THEN 1 ELSE 0 END) AS roas_aberrant
-- 💡 Indice 5 : SUM(CASE WHEN ctr > 0.20 THEN 1 ELSE 0 END) AS ctr_incoherent


In [ ]:
%%sql df_diag_camp <<
-- 📝 TODO : Détecter les campagnes avec budget dépassé et calculer le % sous-performantes
-- 💡 Indice 1 : SUM(CASE WHEN budget_depense_fcfa > budget_alloue_fcfa THEN 1 ELSE 0 END)
-- 💡 Indice 2 : ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)


In [ ]:
%%sql df_diag_conv <<
-- 📝 TODO : Analyser les anomalies dans conversions : achats sans valeur, délais aberrants, distribution par type
-- 💡 Indice 1 : Filtrer sur type_conversion = 'Achat' ET valeur_fcfa IS NULL
-- 💡 Indice 2 : Délai aberrant = delai_clic_conversion_h > 168 (7 jours)
-- 💡 Indice 3 : SUM(CASE WHEN type_conversion = 'Achat' THEN 1 ELSE 0 END)


### 🧠 Tes observations

> - Combien de lignes avec clics > impressions as-tu trouvé ? Quelle en est la cause probable ?
> - Quel pourcentage de campagnes sont sous-performantes ? Que représente ce chiffre en termes de budget gaspillé ?
> - Pourquoi remplace-t-on les revenus négatifs par NULL plutôt que par 0 ?


---
## 4. KPI 1 — Vue globale

### MÉTHODE
Calcule tous les KPIs en une seule requête pour garantir la cohérence. **Filtre obligatoire** : exclure les anomalies identifiées.

> - `CTR` = clics / impressions
> - `CPC` = dépenses / clics
> - `CPA` = dépenses / conversions
> - `ROAS` = revenu / dépenses (**ROAS = 1 = seuil de rentabilité**)
> - `NULLIF(x, 0)` protège les divisions par zéro

In [ ]:
%%sql df_kpi_global <<
-- 📝 TODO : Calculer les KPIs globaux : nb_campagnes, total_impressions, total_clics, total_conversions, CTR%, CPC, CPA, ROAS, % sous-perf
-- 💡 Indice 1 : JOIN campagnes c ON p.campagne_id = c.campagne_id
-- 💡 Indice 2 : Exclure les anomalies : clics <= impressions AND revenu >= 0 AND roas <= 50 AND ctr <= 0.20
-- 💡 Indice 3 : NULLIF(SUM(impressions), 0) pour éviter la division par zéro
-- 💡 Indice 4 : ROUND(SUM(clics) * 100.0 / NULLIF(SUM(impressions), 0), 2) AS ctr_pct


### 🧠 Tes observations

> - Quel est le ROAS moyen ? Comment interprètes-tu cette valeur dans le contexte de l'agence ?
> - Le CTR moyen est de ~2.86%. Qu'est-ce que cela signifie concrètement pour 1 000 impressions ?
> - 26.3% de campagnes sous-performantes — estime le coût mensuel de cette sous-performance (budget total × 26.3% × 70%).


---
## 5. KPI 2 — Performance par canal avec RANK()

### MÉTHODE
`RANK() OVER (ORDER BY roas DESC)` classe les canaux sans réduire le nombre de lignes. Utilise une CTE pour isoler les agrégats, puis applique les rangs dans la requête externe.

In [ ]:
%%sql df_kpi_canal <<
-- 📝 TODO : Calculer les KPIs par canal + 3 classements simultanés avec RANK()
-- 💡 Indice 1 : CTE perf_canal : GROUP BY canal avec ROAS, CTR, CPC, CPA, % sous-perf
-- 💡 Indice 2 : RANK() OVER (ORDER BY roas_moyen DESC) AS rang_roas
-- 💡 Indice 3 : RANK() OVER (ORDER BY cpc_fcfa ASC) AS rang_cout
-- 💡 Indice 4 : RANK() OVER (ORDER BY pct_sous_perf ASC) AS rang_fiabilite
-- 💡 Indice 5 : Ajouter part du budget : ROUND(depenses * 100.0 / SUM(depenses) OVER(), 1)


### 🧠 Tes observations

> - Quel canal a le meilleur ROAS ? Le meilleur CTR ? Sont-ce les mêmes ?
> - Pourquoi Email a-t-il 0% de sous-performance ? Explique la logique métier.
> - Facebook a le CPC le plus élevé — est-ce forcément un mauvais signe ? Dans quels cas Facebook reste-t-il pertinent ?


---
## 6. KPI 3 — Performance par client agence

### MÉTHODE
Jointure `performances_daily` → `campagnes` → `clients_agence` en une requête. `RANK()` sur le ROAS permet d'identifier les clients dont les campagnes performent le mieux.

In [ ]:
%%sql df_kpi_client <<
-- 📝 TODO : KPIs ROAS / CTR / % sous-perf par client agence + RANK() sur ROAS
-- 💡 Indice 1 : JOIN campagnes c ON p.campagne_id = c.campagne_id
-- 💡 Indice 2 : JOIN clients_agence ca ON c.client_id = ca.client_id
-- 💡 Indice 3 : GROUP BY ca.client_id, ca.nom, ca.secteur
-- 💡 Indice 4 : RANK() OVER (ORDER BY ROAS DESC) AS rang_roas


### 🧠 Tes observations

> - Quel secteur concentre les meilleurs ROAS ? Quelle logique marketing explique cela ?
> - Quels clients ont un % sous-perf > 40% ? Que recommanderais-tu à leur account manager ?


---
## 7. KPI 4 — Évolution mensuelle avec LAG()

### MÉTHODE
`LAG(roas) OVER (ORDER BY mois)` retourne la valeur du mois précédent. `strftime(date_perf, '%Y-%m')` extrait l'année-mois. La variation `roas - LAG(roas)` mesure si la performance s'améliore ou se dégrade.

In [ ]:
%%sql df_mensuel <<
-- 📝 TODO : Tendance mensuelle avec LAG() : ROAS, CTR, variation mois sur mois
-- 💡 Indice 1 : strftime(p.date_perf, '%Y-%m') AS mois
-- 💡 Indice 2 : LAG(roas) OVER (ORDER BY mois) AS roas_mois_prec
-- 💡 Indice 3 : ROUND(roas - LAG(roas) OVER (ORDER BY mois), 2) AS variation_roas
-- 💡 Indice 4 : CTE d'abord pour calculer les agrégats mensuels, puis LAG dans la requête externe


### 🧠 Tes observations

> - Identifie les mois avec la plus forte variation positive et négative de ROAS.
> - Y a-t-il des mois de déclin consécutif (`variation_roas` négatif 2 fois de suite) ? Quelle action cela devrait-il déclencher ?
> - Le ROAS varie-t-il plus en été ou en hiver ? Y a-t-il un pattern saisonnier ?


---
## 8. KPI 5 — Top et Bottom campagnes

### MÉTHODE
CTE d'agrégation par `campagne_id`, puis jointure avec `campagnes` et `clients_agence`. `RANK()` permet d'identifier les deux extrêmes sans double requête.

In [ ]:
%%sql df_top_camp <<
-- 📝 TODO : Top 10 campagnes par ROAS : agrégation par campagne_id + jointure clients
-- 💡 Indice 1 : CTE camp_perf : SUM dépenses, ROUND ROAS, CTR, nb_jours
-- 💡 Indice 2 : JOIN campagnes c et clients_agence ca
-- 💡 Indice 3 : RANK() OVER (ORDER BY roas_final DESC) AS rang
-- 💡 Indice 4 : LIMIT 10


In [ ]:
%%sql df_bottom_camp <<
-- 📝 TODO : 10 pires campagnes sous-performantes triées par dépenses décroissantes
-- 💡 Indice 1 : WHERE c.campagne_sous_performe = 1
-- 💡 Indice 2 : ORDER BY cp.depenses_fcfa DESC


### 🧠 Tes observations

> - Quels canaux dominent le Top 10 ? Quel objectif de campagne est le plus représenté ?
> - Les pires campagnes ont-elles des durées plus longues que la moyenne ? Que suggère-t-on à l'account manager ?
> - Compare le ROAS J3 (roas_j3 de marketpulse_analytics) des top vs bottom campagnes — y a-t-il un signal précoce ?


---
## 9. Visualisation 2×2

### MÉTHODE
4 visuels en une figure. Utilise `df_kpi_canal`, `df_mensuel` et `df_an` (marketpulse_analytics) pour construire les graphiques.

In [ ]:
# 📝 TODO : Construire un figure 2×2 avec :
#   ax1 (haut-gauche) : barres ROAS par canal + courbe CTR (axe secondaire)
#   ax2 (haut-droite) : barres horizontales % sous-perf par canal
#   ax3 (bas-gauche)  : barres nb campagnes par canal et statut sous-perf
#   ax4 (bas-droite)  : courbe ROAS mensuel avec ligne de référence 3.31×

# 💡 Indice 1 : fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# 💡 Indice 2 : ax.twinx() pour un axe secondaire
# 💡 Indice 3 : plt.savefig(f'{SAVE_PATH}marketpulse_vue_ensemble.png', dpi=150)

# [Ton code ici]


### 🧠 Tes observations

> - La courbe ROAS mensuelle montre-t-elle une tendance globale sur 24 mois ?
> - Le graphique de sous-performance par canal confirme-t-il tes intuitions des requêtes SQL ?
> - Quelle visualisation t'a le plus aidé à comprendre les données ? Pourquoi ?


---
## Bilan du Notebook 1

### Anomalies à corriger en NB2

| Anomalie | Nb | Table | Traitement NB2 |
|---|---|---|---|
| Clics > impressions | **?** | `performances_daily` | ✍️ À compléter |
| Revenu généré négatif | **?** | `performances_daily` | ✍️ À compléter |
| ROAS aberrant (> 50) | **?** | `performances_daily` | ✍️ À compléter |
| CTR incohérent (> 20%) | **?** | `performances_daily` | ✍️ À compléter |
| Budget dépensé > alloué | **?** | `campagnes` | ✍️ À compléter |
| Valeur conversion NULL | **?** | `conversions` | ✍️ À compléter |

### KPIs de référence (complète après exécution)

| KPI | Valeur |
|---|---|
| ROAS moyen global | **___×** |
| CTR moyen global | **___%** |
| CPC moyen global | **___ FCFA** |
| Campagnes sous-perf | **___** (___%) |
| Canal le plus efficient | **___** |
| Canal le plus risqué | **___** |


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.